**Problem Statement:**

Despite significant research activity on anomaly-based IDS, critical gaps remain in the literature that limit the practical applicability of proposed systems. Most existing studies evaluate a single model in isolation, without rigorous statistical comparison across multiple ML and DL approaches on the same standardized test set. Class imbalance particularly extreme for rare attacks such as Heartbleed, which has only 11 samples in CICIDS-2017 is frequently ignored or handled inconsistently, leading to inflated accuracy metrics that mask poor detection rates for the most dangerous attacks. Furthermore, the majority of published models operate as black boxes: they report accuracy figures without explaining which network flow features drive their decisions, making the models untrustworthy from an operational security perspective. Finally, temporal generalisation whether a model trained on Monday's traffic can detect Friday's attacks  is rarely evaluated, leaving a critical question about real-world concept drift unanswered. This project addresses all four of these gaps through a rigorous, reproducible, and explainable experimental framework applied to the CICIDS-2017 dataset.


**Objective :**

This study aims to design, implement, evaluate, and compare a suite of anomaly-based IDS models spanning unsupervised (Isolation Forest), supervised classical (Random Forest, XGBoost), and deep learning (Long Short-Term Memory [LSTM] Autoencoder) approaches that are applied to the CICIDS-2017 network traffic dataset. Specifically, the study seeks to:
*  classify network flows as either benign or one of 14 attack types;
*  quantify the impact of class imbalance handling techniques on rare-attack detection;
*  explain model decisions through SHapley Additive exPlanations (SHAP)-based feature importance analysis;
*  evaluate cross-day generalisation to assess the presence and magnitude of concept drift in network intrusion data.


**Data Dictionary :**

Feature Name : Description
* Flow Duration: Total duration of the bidirectional flow in microseconds
* Fwd Packet Length Mean: Mean size (bytes) of packets sent in the forward direction
* Bwd Packet Length Mean:Mean size (bytes) of packets received in the backward direction
* Flow Bytes/s: Total bytes transferred per second; high values signal DoS/DDoS
* Flow Packets/s: Total packets per second; combined with duration flags floods
* Flow IAT Mean: Mean inter-arrival time between packets; regularity signals botnet C2
* RST Flag Count: Count of TCP RST flags; high count signals PortScan activity
* SYN Flag Count: Count of TCP SYN flags; excessive SYN indicates SYN flood
* Init Win Bytes Fwd : Initial TCP window size; anomalous values flag Slowloris
* Fwd Packets/s: Forward direction packet rate; primary Brute Force signal
* Bwd Packet Length Max: Maximum backward packet size; large values signal data exfiltration
* Label : Categorical :15 classes :Ground truth: BENIGN or named attack type



**Importing the necessary libraries :**

In [1]:
# Install a numpy version known to be compatible with many gensim/tensorflow 2.x setups
!pip install numpy==1.24.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 59.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [2]:
# installing libraries to remove accented characters and use word embeddings
!pip install unidecode gensim -q
#!pip install --user unidecode gensim -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 80.9 MB/s eta 0:00:00


In [3]:
!pip install zeugma==0.41

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 4.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.4-py3-none-any.whl (314 kB)
  Created wheel for zeugma: filename=zeugma-0.41-py3-none-any.whl size=9857 sha256=cd4cf5b1cfbd62a85cd08478fd55fa0ce3426651cd77654ae45d25c72c4f7121
  Stored in directory: /root/.cache/pip/wheels/3d/42/7b/2da6f83151576f69aa17d62ec011597c5e700b253a3b9fb7ec
  Created wheel for fastText: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653907 sha256=54a845778d5fac0f763ecd099502951f75e6a17742aa85ba0b52eb469e3a0900
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built zeugma fastText


In [4]:
# installing libraries to remove accented characters and use word embeddings
!pip install unidecode gensim -q
#!pip install --user unidecode gensim -q

In [5]:
!pip install --upgrade numpy pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 130.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 122.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pan

In [1]:
# Numerical libraries
import numpy as np
# to handle data in form of rows and columns
import pandas as pd
import time

# importing ploting libraries
import matplotlib.pyplot as plt

#importing seaborn for statistical plots
import seaborn as sns

# to use regular expressions for manipulating text data
import re

# to load the natural language toolkit
import nltk
nltk.download('stopwords')    # loading the stopwords
nltk.download('wordnet')    # loading the wordnet module that is used in stemming

# to remove common stop words
from nltk.corpus import stopwords

# to perform stemming
from nltk.stem.porter import PorterStemmer

# to create Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

# to split data into train and test sets
from sklearn.model_selection import train_test_split

# to build a Random Forest model
from sklearn.ensemble import RandomForestClassifier

# to compute metrics to evaluate the model
from sklearn import metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# To tune different models
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV


# To encode the target variable
from sklearn.preprocessing import LabelEncoder

# Libraries different ensemble classifiers
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    StackingClassifier,
)

from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier

#To build model for prediction
#logistic Regression
from sklearn.linear_model import LogisticRegression
#for KNN & Naive Bayes
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
#for DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier
#for DecisionTreeClassifier
from sklearn.svm import SVC

# To compute metrics to evaluate the model
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_score, recall_score, classification_report

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
# Converting the Stanford GloVe model vector format to word2vec
from gensim.scripts.glove2word2vec import glove2word2vec
from gensim.models import KeyedVectors

**Loading the dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import glob
import os

# ── Option 1: If all 7 files are in the same folder ──────────────
folder_path = "/content/drive/MyDrive/QM 640 MS Project"
all_files = glob.glob(os.path.join(folder_path, "*.csv"))

df_list = []
for f in all_files:
    temp = pd.read_csv(f, low_memory=False)
    temp.columns = temp.columns.str.strip()   # remove leading/trailing spaces
    temp['source_file'] = os.path.basename(f) # optional: track which file each row came from
    df_list.append(temp)
    print(f"Loaded: {os.path.basename(f)} — {len(temp):,} rows")

merged = pd.concat(df_list, ignore_index=True)

print(f"\nTotal rows after merge: {len(merged):,}")
print(f"Total columns:          {merged.shape[1]}")
print(f"Label distribution:\n{merged['Label'].value_counts()}")

# Save
merged.to_csv("CICIDS2017_merged.csv", index=False)
print("\nSaved: CICIDS2017_merged.csv")

In [ ]:
df= merged.copy()

In [ ]:
df.tail(20)

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
#Lets define a function to fetch the Unique, min & max & count of value in a categorical column & count of unique values in a categorical column

def PrintMinMaxUniqueVals(df):
  '''
  Function to print no of unique values, min & max for every attribute
  Input : df - pandas dataframe
  returns: Nothing
  '''

  for col in df.columns:
    if df[col].dtype =='object': #and col !='Critical Risk':
      print('\n\nNo.of unique values for attribute {0} is {1} and the values are : {2}'.format( col, df[col].nunique(), df[col].unique()))
    else:
      print('\n\nFor attribute {0} no.of unique values is {1}, min is {2}, max is {3}'.format( col, df[col].nunique(), df[col].min(), df[col].max()))
  #Making a list of all categorical variables
  return

#Printing number of count of each unique value in each column
def PrintCountUniqueVals(df):
  for col in df.columns:
    if df[col].dtype =='object' and col !='Label':
      print('\n\nCount of unique values for attribute {1} '.format(col,df[col].value_counts()))
      print("-"*40)

  return


#Lets define a function to fetch the duplicate and missing value in the column

def VerifyDuplicatesMissingVals(df):
  '''
  Function to fetch Missing and Duplicate values in each attribute
  Input - df -pandas dataframe
  returns Nothing
  '''
  print('\n\nNo.of duplicate rows present in the dataset is {}'.format(df.duplicated().sum()))

  print('\n\nNo.of missing values present in the dataset is \n\n{}'.format(df.isnull().sum()))
  return

Checking for duplicate values:

In [ ]:
# checking for duplicate values
df.duplicated().sum()

In [ ]:
PrintMinMaxUniqueVals(df)

In [ ]:
PrintCountUniqueVals(df)

**Checking for Datatype Description :**

In [ ]:
df.describe()

In [ ]:
from pandas.core.frame import DataFrame
total_dups = df.duplicated().sum()
print(f"\nTotal duplicate rows: {total_dups:,}")
# See which Labels the duplicates belong to
dup_rows = df[df.duplicated(keep=False)]
print(f"\nDuplicate rows by Label:")
print(dup_rows['Label'].str.strip().value_counts())

# Near-duplicates — same features, different label (dangerous!)
feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()
near_dups = df.duplicated(subset=feature_cols, keep=False)
print(f"Near-duplicates (same features, possibly diff label): {near_dups.sum():,}")


# Show cases where same features have DIFFERENT labels — data conflict
conflicts = df[df.duplicated(subset=feature_cols, keep=False)]
conflict_labels = conflicts.groupby(feature_cols[:5])['Label'].nunique()
multi_label = conflict_labels[conflict_labels > 1]
print(f"Feature rows with conflicting labels: {len(multi_label):,}")

**Data Pre-processing**

In [ ]:
# Data Cleaning: Handle infinite values, negative durations, and fill NaNs

# Check how many infinite values exist
inf_mask = np.isinf(df.select_dtypes(include=[np.number]))
print("Infinite values per column:")
print(inf_mask.sum()[inf_mask.sum() > 0])

print(f"\nRows before removing Inf-turned-NaN: {len(df):,}")

# Replace Inf with NaN — then drop those rows
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
print(f"\nRows after removing Inf-turned-NaN: {len(df):,}")


In [ ]:
# Replace inf and -inf with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Replace negative Flow Duration with NaN
df.loc[df['Flow Duration'] < 0, 'Flow Duration'] = np.nan

# Replace negative Flow IAT Mean with NaN
df.loc[df['Flow IAT Mean'] < 0, 'Flow IAT Mean'] = np.nan

# Replace negative Init_Win_bytes_forward with NaN
df.loc[df['Init_Win_bytes_forward'] < 0, 'Init_Win_bytes_forward'] = np.nan

# Fill all remaining NaNs with 0
df.fillna(0, inplace=True)

print("Data cleaning complete: Infinite values and problematic negative values handled, NaNs filled with 0.")
print(f"\nRows after removing Inf-turned-NaN: {len(df):,}")

**Exploratory Data Analysis (EDA)**

In [ ]:
# function to create labeled barplots

def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 1, 5))
    else:
        plt.figure(figsize=(n + 1, 5))

    plt.xticks(rotation=90, fontsize=15)
    order = data[feature].value_counts().index[:n]

    plt.xticks(rotation=90, fontsize=15)

    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        hue=feature,  # Explicitly set hue to match the x-axis variable
        order=order,
        legend=False # This hides the redundant legend
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",

        )  # annotate the percentage

    plt.show()  # show the plot


In [ ]:
###function to plot histogram & boxplot
def histogram_boxplot(data, feature, figsize=(15,10), kde=False, bins=None):
  """
  Boxplot and histogram combined

  data: pd dataframe
  feature: dataframe column
  figsize: size of figure (deafult (15,10))
  kde: whether to show the density curve (default False)
  bins: number of bins for histogram  (default None)
  """
  f2, (ax_box2, ax_hist2) = plt.subplots(
      nrows =2, # number of rows of the subplot grid =2
      sharex=True, # x-axis will be shaed among all subplots
      gridspec_kw={"height_ratios":(0.25,0.75)},
      figsize = figsize,
  ) # creating the 2 subplots
  sns.boxplot(
      data =data, x=feature,ax=ax_box2, showmeans= True, color="violet"
  ) # boxplot will be ceated and a triangle will indicate the mean value of the column
  sns.histplot(
      data =data, x=feature, ax =ax_hist2,bins =bins,kde=kde
  )if bins else sns.histplot(
      data =data, x=feature, ax =ax_hist2,kde =kde
  ) #for histogram
  ax_hist2.axvline(
      data[feature].mean(), color="green", linestyle="--"
  ) # Add mean to the histogram
  ax_hist2.axvline(
      data[feature].median(),color="black", linestyle="-"
  ) #Add  median to the histogram


In [ ]:
def plot(col, title, palette, edgecolor):
    # Precompute counts once, keep order consistent across plots
    vc = ind_data[col].value_counts(dropna=False)

    # If you truly wanted "the 2nd most common" by position, use iloc
    value = vc.iloc[1] if len(vc) > 1 else float('nan')

    plt.figure(figsize=(20, 15))

    # --- Bar + line (same axes) ---
    ax = plt.subplot(2, 2, 1)

    # seaborn 0.14+: pass x via keyword; using hue=col allows palette without warning
    sns.countplot(
        data=ind_data,
        x=col,
        order=vc.index,
        hue=col,            # <- required to use palette without deprecation
        palette=palette,
        edgecolor=edgecolor,
        legend=False
    )

    # Overlay line using the same x order; use color (not palette) on single line
    sns.lineplot(
        x=vc.index.astype(str),
        y=vc.values,
        marker='o',
        sort=False,         # keep the given order
        color='black',
        ax=ax
    )

    plt.title(title)

    # --- Pie chart ---
    plt.subplot(2, 2, 2)
    plt.pie(
        vc.values,
        labels=vc.index.astype(str),
        autopct="%.2f%%",   # matplotlib formats % automatically
        shadow=True,
        explode=[0.1] * len(vc),
        startangle=-135
    )
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
df.columns

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

labeled_barplot(df, "Label", perc=True)

In [ ]:
import holoviews as hv
hv.extension("bokeh", "matplotlib")
from holoviews import opts
cr_risk_cnt = np.round(df['Label'].value_counts(normalize=True) * 100)

hv.Bars(cr_risk_cnt[::-1]).opts(title="Label", color="#8888ff", xlabel="Label", ylabel="Percentage", xformatter='%d%%')\
                .opts(opts.Bars(width=600, height=600,tools=['hover'],show_grid=True,invert_axes=True))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv("CICIDS2017_merged.csv", low_memory=False)
df.columns = df.columns.str.strip()
df['Label'] = df['Label'].str.strip()

# ── Bar chart: class distribution ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full count
label_counts = df['Label'].value_counts()
label_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title("Class Distribution — All Labels", fontsize=14)
axes[0].set_xlabel("Attack Category")
axes[0].set_ylabel("Number of Flows")
axes[0].tick_params(axis='x', rotation=45)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=8)

# Log scale — shows minority classes clearly
label_counts.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title("Class Distribution — Log Scale", fontsize=14)
axes[1].set_xlabel("Attack Category")
axes[1].set_ylabel("Number of Flows (log scale)")
axes[1].set_yscale('log')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("EDA_01_class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

# Print imbalance ratio
majority = label_counts.max()
minority = label_counts.min()
print(f"Imbalance ratio (majority/minority): {majority/minority:.0f}:1")
print(f"\nClass percentages:\n{(label_counts/len(df)*100).round(2)}")

In [ ]:
# ── The 8 most important features for univariate analysis ──────────
key_features = [
    'Flow Duration',
    'Flow Bytes/s',
    'Flow Packets/s',
    'Fwd Packet Length Mean',
    'Bwd Packet Length Mean',
    'Flow IAT Mean',
    'RST Flag Count',
    'Init_Win_bytes_forward'
]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    if feat in df.columns:
        feature_data = df[feat] # Use df instead of data
        # Use log1p to handle skewed distributions, after ensuring no non-positive values
        # Add a small constant to avoid log(0) if any values are 0 after cleaning
        sns.histplot(np.log1p(feature_data.dropna()), bins=60, kde=True,
                     ax=axes[i], color='steelblue', alpha=0.7)
        axes[i].set_title(f'{feat}\n(log1p scale)', fontsize=10)
        axes[i].set_xlabel('')
        # Add descriptive stats as text
        axes[i].text(0.98, 0.95,
                     f'Mean: {feature_data.mean():.2f}\nStd: {feature_data.std():.2f}\nSkew: {feature_data.skew():.2f}',
                     transform=axes[i].transAxes, ha='right', va='top',
                     fontsize=8, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle("Univariate Distributions — Key Network Flow Features", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("EDA_02_univariate_distributions.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary statistics for thesis Table ─────────────────────────────
stats = df[key_features].describe().T # Use df instead of data
stats['skewness'] = df[key_features].skew() # Use df instead of data
stats['kurtosis'] = df[key_features].kurt() # Use df instead of data
stats = stats[['mean', 'std', 'min', '25%', '50%', '75%', 'max', 'skewness', 'kurtosis']]
stats.columns = ['Mean', 'Std Dev', 'Min', 'Q1', 'Median', 'Q3', 'Max', 'Skewness', 'Kurtosis']
print(stats.round(3).to_string())
stats.to_csv("EDA_descriptive_stats.csv")

In [ ]:
# ── Box plots — shows outliers clearly ─────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 6))

box_features = ['Flow Duration', 'Flow Bytes/s',
                 'Fwd Packet Length Mean', 'Flow IAT Mean']

for i, feat in enumerate(box_features):
    if feat in df.columns:
        # Sample for speed
        feature_data = df[feat].sample(min(50000, len(df)), random_state=42) # Use df instead of data
        axes[i].boxplot(feature_data.dropna(), vert=True, patch_artist=True,
                        boxprops=dict(facecolor='lightblue'),
                        medianprops=dict(color='red', linewidth=2))
        axes[i].set_title(feat, fontsize=10)
        axes[i].set_ylabel("Value")

plt.suptitle("Box Plots — Outlier Detection in Key Features", fontsize=14)
plt.tight_layout()
plt.savefig("EDA_03_boxplots.png", dpi=150, bbox_inches='tight')
plt.show()

**Bivariate Analysis**

In [ ]:
# ── These 6 features show the clearest attack vs benign separation ──
bivariate_features = [
    'Flow Bytes/s',
    'Flow Packets/s',
    'RST Flag Count',
    'Fwd Packet Length Mean',
    'Bwd Packet Length Mean',
    'Flow IAT Mean'
]

# Binary label for cleaner plots
df['Label_binary'] = df['Label'].apply(lambda x: 'BENIGN' if x == 'BENIGN' else 'ATTACK')

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, feat in enumerate(bivariate_features):
    if feat in df.columns:
        # Sample for speed
        sample = df.sample(min(30000, len(df)), random_state=42)
        sample[feat+'_log'] = np.log1p(sample[feat].clip(lower=0))

        sns.violinplot(data=sample, x='Label_binary', y=feat+'_log',
                       palette={'BENIGN': '#2E86AB', 'ATTACK': '#E84855'},
                       ax=axes[i], inner='box')
        axes[i].set_title(f'{feat}\nvs Attack/Benign', fontsize=10)
        axes[i].set_xlabel('')
        axes[i].set_ylabel('log1p(value)')

plt.suptitle("Bivariate Analysis — Feature Distributions by Class (Benign vs Attack)",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("EDA_04_violin_bivariate.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Box plot for each of the 15 attack types ───────────────────────
# Use Flow Bytes/s — the single most discriminative feature
fig, ax = plt.subplots(figsize=(18, 7))

# Keep top 8 classes by count for readability
top_classes = df['Label'].value_counts().head(8).index.tolist()
df_top = df[df['Label'].isin(top_classes)].copy()
df_top['log_bytes'] = np.log1p(df_top['Flow Bytes/s'].clip(lower=0))

sns.boxplot(data=df_top, x='Label', y='log_bytes',
            palette='Set2', ax=ax, showfliers=False)
ax.set_title("Flow Bytes/s Distribution by Attack Category\n(log scale, no outliers shown)",
             fontsize=13)
ax.set_xlabel("Attack Category")
ax.set_ylabel("log1p(Flow Bytes/s)")
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("EDA_05_multiclass_boxplot.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation heatmap — top 15 most variable features ───────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if 'label' not in c.lower()]

# Select top 15 by variance for a readable heatmap
top_var_cols = df[numeric_cols].var().nlargest(15).index.tolist()

corr = df[top_var_cols].corr()

fig, ax = plt.subplots(figsize=(12, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))   # show lower triangle only
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5,
            annot_kws={'size': 8})
ax.set_title("Feature Correlation Heatmap — Top 15 High-Variance Features",
             fontsize=13)
plt.tight_layout()
plt.savefig("EDA_06_correlation_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

# Print highly correlated pairs (|r| > 0.9)
print("\nHighly correlated feature pairs (|r| > 0.90):")
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.90:
            print(f"  {corr.columns[i]} ↔ {corr.columns[j]}: r = {corr.iloc[i,j]:.3f}")

From the heat map we can see the highly correlated features which can be removed.

In [ ]:
# ── Scatter: two most important features, coloured by binary label ──
sample = df.sample(min(20000, len(df)), random_state=42).copy()
sample['log_dur']   = np.log1p(sample['Flow Duration'].clip(lower=0))
sample['log_bytes'] = np.log1p(sample['Flow Bytes/s'].clip(lower=0))

fig, ax = plt.subplots(figsize=(12, 7))
colors = {'BENIGN': '#2E86AB', 'ATTACK': '#E84855'}
for label, grp in sample.groupby('Label_binary'):
    ax.scatter(grp['log_dur'], grp['log_bytes'],
               c=colors[label], label=label,
               alpha=0.3, s=8, rasterized=True)

ax.set_xlabel("log1p(Flow Duration)")
ax.set_ylabel("log1p(Flow Bytes/s)")
ax.set_title("Scatter Plot: Flow Duration vs Flow Bytes/s\nColoured by Class")
ax.legend(markerscale=4)
plt.tight_layout()
plt.savefig("EDA_07_scatter_bivariate.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Pairplot — best 5 features for thesis EDA section ──────────────
pairplot_features = [
    'Flow Bytes/s', 'Flow Packets/s',
    'Fwd Packet Length Mean', 'RST Flag Count', 'Flow IAT Mean',
    'Label_binary'
]
sample = df[pairplot_features].sample(min(5000, len(df)), random_state=42).copy()
for f in pairplot_features[:-1]:
    sample[f] = np.log1p(sample[f].clip(lower=0))

g = sns.pairplot(sample, hue='Label_binary',
                 palette={'BENIGN': '#2E86AB', 'ATTACK': '#E84855'},
                 plot_kws={'alpha': 0.3, 's': 10},
                 diag_kind='kde')
g.fig.suptitle("Pairplot — Top 5 Features by Class (log scale)", y=1.02, fontsize=13)
plt.savefig("EDA_08_pairplot.png", dpi=150, bbox_inches='tight')
plt.show()

**Dataset Pre-processing**
From the heat map I found 7 highly correlated featues,the list is as below. I will drop the correlated features.
Highly correlated feature pairs (|r| > 0.90):
  Flow Duration ↔ Fwd IAT Total: r = 0.999
  Fwd IAT Max ↔ Flow IAT Max: r = 0.998
  Fwd IAT Max ↔ Idle Max: r = 0.988
  Fwd IAT Max ↔ Idle Mean: r = 0.978
  Fwd IAT Max ↔ Idle Min: r = 0.949
  Fwd IAT Max ↔ Fwd IAT Std: r = 0.918
  Flow IAT Max ↔ Idle Max: r = 0.989
  Flow IAT Max ↔ Idle Mean: r = 0.980
  Flow IAT Max ↔ Idle Min: r = 0.951
  Flow IAT Max ↔ Fwd IAT Std: r = 0.916
  Idle Max ↔ Idle Mean: r = 0.990
  Idle Max ↔ Idle Min: r = 0.962
  Idle Max ↔ Fwd IAT Std: r = 0.915
  Idle Mean ↔ Idle Min: r = 0.990
  Idle Mean ↔ Fwd IAT Std: r = 0.905
  Fwd Header Length ↔ Fwd Header Length.1: r = 1.000

In [ ]:
#Removing highly correalted features.
print(f"Shape before dropping: {df.shape}")

# ── Features to drop based on your heatmap results ────────────────

drop_features = [
    # Perfect duplicate — r = 1.000
    'Fwd Header Length.1',     # exact copy of Fwd Header Length

    # Near-perfect correlation with Flow Duration — r = 0.999
    'Fwd IAT Total',           # keeps: Flow Duration

    # Near-perfect correlation with Fwd IAT Max — r = 0.998
    'Flow IAT Max',            # keeps: Fwd IAT Max

    # Idle group — all highly correlated with Fwd IAT Max
    'Idle Max',                # r = 0.988 with Fwd IAT Max
    'Idle Mean',               # r = 0.978 with Fwd IAT Max
    'Idle Min',                # r = 0.949 with Fwd IAT Max
    'Fwd IAT Std',             # r = 0.918 with Fwd IAT Max
]

# Only drop columns that actually exist in the dataframe
existing_drops = [c for c in drop_features if c in df.columns]
missing = [c for c in drop_features if c not in df.columns]

if missing:
    print(f"Warning — not found in dataframe: {missing}")

df.drop(columns=existing_drops, inplace=True)
print(f"Shape after dropping:  {df.shape}")
print(f"Features removed: {len(existing_drops)}")
print(f"Features removed list: {existing_drops}")

**Verify the correlation is gone after dropping**


In [ ]:
# Confirm no pairs above 0.90 remain
numeric_cols = df.select_dtypes(include=[np.number]).columns
numeric_cols = [c for c in numeric_cols if 'label' not in c.lower()]

corr = df[numeric_cols].corr().abs()
upper = corr.where(np.triu(np.ones_like(corr, dtype=bool), k=1))

remaining_high = []
for col in upper.columns:
    for idx in upper.index:
        val = upper.loc[idx, col]
        if pd.notna(val) and val > 0.90:
            remaining_high.append((idx, col, round(val, 3)))

if remaining_high:
    print(f"\nStill correlated above 0.90:")
    for a, b, r in remaining_high:
        print(f"  {a} ↔ {b}: r = {r}")
else:
    print("\nNo pairs above 0.90 remain — correlation clean!")

In [ ]:
# These features LOOK similar but serve different purposes — KEEP them
keep_reasoning = {
    'Flow Duration':     'Primary time feature — most interpretable for analysts',
    'Fwd IAT Max':       'Single strongest idle/timing signal — captures botnet beaconing',
    'Fwd Header Length': 'Real column — keep original, drop the .1 duplicate only',
    'Idle Max':          'Dropped — Fwd IAT Max already captures this',
}

print("\nFeatures retained and why:")
for feat, reason in keep_reasoning.items():
    status = "KEPT" if feat not in drop_features else "DROPPED"
    print(f"  [{status}] {feat}: {reason}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# COMPLETE DROP LIST — based on your correlation results
# ══════════════════════════════════════════════════════════════════
drop_second_round = [

    # Group 1 — Packet count cluster
    'Total Length of Bwd Packets',   # r=0.994 with Total Backward Packets
    'Subflow Fwd Packets',           # r=1.000 with Total Fwd Packets
    'Subflow Bwd Packets',           # r=1.000 with Total Backward Packets
    'Subflow Bwd Bytes',             # r=1.000 with Total Length of Bwd Packets

    # Group 2 — Subflow bytes duplicate
    'Subflow Fwd Bytes',             # r=1.000 with Total Length of Fwd Packets

    # Group 3 — Backward packet length cluster
    'Bwd Packet Length Max',         # r=0.959 with Bwd Packet Length Mean
    'Bwd Packet Length Std',         # r=0.942 with Bwd Packet Length Mean
    'Avg Bwd Segment Size',          # r=1.000 with Bwd Packet Length Mean

    # Group 4 — Global packet length cluster
    'Max Packet Length',             # r=0.984 with Packet Length Std
    'Packet Length Std',             # r=0.944 with Packet Length Mean
    'Packet Length Variance',        # r=0.925 with Packet Length Std
    'Average Packet Size',           # r=0.998 with Packet Length Mean
    'Avg Fwd Segment Size',          # r=1.000 with Fwd Packet Length Mean

    # Group 5 — Forward packet length
    'Fwd Packet Length Std',         # r=0.968 with Fwd Packet Length Max

    # Group 6 — Flag exact duplicates
    'Fwd PSH Flags',                 # r=1.000 with SYN Flag Count
    'Fwd URG Flags',                 # r=1.000 with CWE Flag Count
    'CWE Flag Count',                # r=1.000 with Fwd URG Flags
    'ECE Flag Count',                # r=0.998 with RST Flag Count

    # Group 7 — Timing / IAT cluster
    'Fwd IAT Mean',                  # r=0.900 with Flow IAT Mean
    'Flow IAT Std',                  # r=0.936 with Fwd IAT Max
    'Bwd IAT Min',                   # r=0.933 with Bwd IAT Mean
    'Fwd Packets/s',                 # r=0.988 with Flow Packets/s

    # Group 8 — Active timing
    'Active Min',                    # r=0.906 with Active Mean
]

# Safe drop — only remove columns that exist
existing = [c for c in drop_second_round if c in df.columns]
missing  = [c for c in drop_second_round if c not in df.columns]

if missing:
    print(f"Not found (already dropped or name mismatch): {missing}")

df.drop(columns=existing, inplace=True)
print(f"Shape after:  {df.shape}")
print(f"Dropped {len(existing)} features in round 2")

In [ ]:
# Final verification pass
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if 'label' not in c.lower()]

corr = df[numeric_cols].corr().abs()
upper = corr.where(np.triu(np.ones_like(corr, dtype=bool), k=1))

still_high = [(i, c, round(upper.loc[i,c], 3))
              for c in upper.columns
              for i in upper.index
              if pd.notna(upper.loc[i,c]) and upper.loc[i,c] > 0.90]

if still_high:
    print(f"\nStill correlated above 0.90 ({len(still_high)} pairs):")
    for a, b, r in still_high:
        print(f"  {a} ↔ {b}: r = {r}")
else:
    print("\nAll correlations below 0.90 — feature selection complete!")

# Save
df.to_csv("CICIDS2017_final_features.csv", index=False)
print(f"\nSaved: CICIDS2017_final_features.csv")
print(f"Final feature count: {len(numeric_cols)} features")

In [ ]:
print(f"Shape before: {df.shape}")

# Drop only Packet Length Mean — keep both packet count columns
df.drop(columns=['Packet Length Mean'], inplace=True, errors='ignore')

print(f"Shape after:  {df.shape}")
print(f"Features remaining: {df.select_dtypes(include='number').shape[1]}")

df.to_csv("CICIDS2017_final_features.csv", index=False)
print("Saved.")

In [ ]:
import numpy as np

numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if 'label' not in c.lower()]

corr  = df[numeric_cols].corr().abs()
upper = corr.where(np.triu(np.ones_like(corr, dtype=bool), k=1))

still_high = [(i, c, round(upper.loc[i, c], 3))
              for c in upper.columns
              for i in upper.index
              if pd.notna(upper.loc[i, c]) and upper.loc[i, c] > 0.90]

if still_high:
    print(f"Still correlated ({len(still_high)} pairs):")
    for a, b, r in still_high:
        print(f"  {a} ↔ {b}: r = {r}")
else:
    print("Feature selection complete — no pairs above 0.90 remain.")
    print(f"Final clean feature count: {len(numeric_cols)}")

These two are nearly identical but they carry different directional meaning in network security:
Total Fwd Packets = packets sent from client → server
Total Backward Packets = packets sent from server → client
In normal HTTP traffic both are balanced. In a DoS attack, Total Fwd Packets explodes while Total Backward Packets stays low (server cannot respond). In a PortScan, Total Fwd Packets is high but Total Backward Packets is near zero (most ports closed — no response). This directional asymmetry is exactly what the anomaly detection models need.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import os

# ── Load your feature-selected cleaned file ───────────────────────
df = pd.read_csv("CICIDS2017_final_features.csv", low_memory=False)
df.columns = df.columns.str.strip()
df['Label'] = df['Label'].str.strip()

print(f"Dataset shape: {df.shape}")
print(f"Label distribution:\n{df['Label'].value_counts()}\n")

# ══════════════════════════════════════════════════════════════════
# LABEL ENCODING
# Do this BEFORE normalisation so label columns are
# excluded from scaling
# ══════════════════════════════════════════════════════════════════

# Binary label — 0 = BENIGN, 1 = ATTACK (for RQ1, RQ2, RQ4)
df['Label_binary'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

# Multiclass label — 0 to 14 (for per-attack-type analysis)
le = LabelEncoder()
df['Label_encoded'] = le.fit_transform(df['Label'])

# Save the label encoder — needed later for SHAP class names
#with open('label_encoder.pkl', 'wb') as f:
   # pickle.dump(le, f)

print("Label encoding complete.")
print(f"Binary labels   — 0: {(df['Label_binary']==0).sum():,}  "
      f"1: {(df['Label_binary']==1).sum():,}")
print(f"Multiclass labels: {list(le.classes_)}\n")

# ══════════════════════════════════════════════════════════════════
# SEPARATE FEATURES FROM LABELS
# Identify all numeric feature columns — exclude label columns
# ══════════════════════════════════════════════════════════════════

label_cols   = ['Label', 'Label_binary', 'Label_encoded']
feature_cols = [c for c in df.columns
                if c not in label_cols
                and df[c].dtype in [np.float64, np.int64, np.float32, np.int32]]

print(f"Feature columns to normalise: {len(feature_cols)}")
print(f"Feature list:\n{feature_cols}\n")

X = df[feature_cols].copy()

# Explicitly handle infinites and NaNs in X right before scaling
# This is crucial as reloading from CSV or previous ops might have left some
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)

y_binary    = df['Label_binary'].values
y_multiclass = df['Label_encoded'].values

# ══════════════════════════════════════════════════════════════════
#TRAIN / TEST SPLIT
# Must be done BEFORE normalisation to prevent data leakage
# Scaler must be fit on training data only
# ══════════════════════════════════════════════════════════════════

# Binary split — used for RQ1, RQ2, RQ4
X_train, X_test, y_train_bin, y_test_bin = train_test_split(
    X, y_binary,
    test_size=0.2,
    stratify=y_binary,       # preserves class ratio in both sets
    random_state=42
)

# Multiclass split — same indices, different label array
_, _, y_train_multi, y_test_multi = train_test_split(
    X, y_multiclass,
    test_size=0.2,
    stratify=y_multiclass,   # preserves all 15 class ratios
    random_state=42          # same seed = same row split
)

print(f"Train size: {len(X_train):,} rows")
print(f"Test size:  {len(X_test):,} rows")
print(f"\nTrain binary distribution:")
unique, counts = np.unique(y_train_bin, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {'BENIGN' if u==0 else 'ATTACK'}: {c:,}")
print(f"\nTest binary distribution:")
unique, counts = np.unique(y_test_bin, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {'BENIGN' if u==0 else 'ATTACK'}: {c:,}")

# Verify minority classes survived the split
print(f"\nMinority class check in test set:")
for label in ['Heartbleed', 'Infiltration', 'Web Attack \u2013 Sql Injection']:
    enc_val = le.transform([label])[0] if label in le.classes_ else None
    if enc_val is not None:
        count = sum(y_test_multi == enc_val)
        print(f"  {label}: {count} samples in test set "
              f"{'OK' if count > 0 else 'WARNING — NONE IN TEST SET'}")

# ══════════════════════════════════════════════════════════════════
# NORMALISATION (Min-Max Scaling)
# Fit scaler on TRAINING data only
# Transform both train and test using the fitted scaler
# This is critical — fitting on test data = data leakage
# ══════════════════════════════════════════════════════════════════

scaler = MinMaxScaler()

# Fit ONLY on training set
X_train_scaled = scaler.fit_transform(X_train)

# Transform test set using training statistics
X_test_scaled  = scaler.transform(X_test)

# Verify range
print(f"\nNormalisation check:")
print(f"  X_train_scaled min: {X_train_scaled.min():.4f}  "
      f"max: {X_train_scaled.max():.4f}")
print(f"  X_test_scaled  min: {X_test_scaled.min():.4f}  "
      f"max: {X_test_scaled.max():.4f}")
# Note: test set may have values slightly outside [0,1]
# due to unseen extreme values — this is expected and correct

In [ ]:
print("PREPROCESSING COMPLETE — FINAL SUMMARY")
print("="*55)
print(f"Total samples:        {len(df):,}")
print(f"Features after selection: {len(feature_cols)}")
print(f"Train set:            {len(X_train):,} ({len(X_train)/len(df)*100:.0f}%)")
print(f"Test set:             {len(X_test):,}  ({len(X_test)/len(df)*100:.0f}%)")
print(f"Scaling:              Min-Max [0, 1] — fit on train only")
print(f"Labels:               Binary (0/1) + Multiclass (0-14)")
print(f"Scaler saved:         minmax_scaler.pkl")
print(f"Encoder saved:        label_encoder.pkl")
print(f"Ready for:            Model training")

In [ ]:
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              roc_auc_score, average_precision_score,
                              classification_report, confusion_matrix,
                              ConfusionMatrixDisplay)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from scipy.stats import wilcoxon
import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from imblearn.over_sampling import SMOTE

In [ ]:
print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")
print(f"Train — Benign: {sum(y_train_bin==0):,}  Attack: {sum(y_train_bin==1):,}")
print(f"Test  — Benign: {sum(y_test_bin==0):,}   Attack: {sum(y_test_bin==1):,}\n")

# ── SMOTE 0rignal — apply to training set only ───────────────────────────
#smote = SMOTE(sampling_strategy= 0.3, k_neighbors=3, random_state=42)
#X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train_bin)
#print(f"After SMOTE — Benign: {sum(y_train_bal==0):,}  "
#      f"Attack: {sum(y_train_bal==1):,}\n")
#------

smote = SMOTE(
    sampling_strategy={cls: 1000
                       for cls, count in zip(*np.unique(y_train_bin,
                                             return_counts=True))
                       if count < 1000},
    k_neighbors=3,
    random_state=42
)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train_bin)
print(f"After SMOTE — Benign: {sum(y_train_bal==0):,}  "
      f"Attack: {sum(y_train_bal==1):,}\n")

# ── Helper: evaluate any model ────────────────────────────────────
def evaluate(name, y_true, y_pred, y_prob=None):
    f1  = f1_score(y_true, y_pred, zero_division=0)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob) if y_prob is not None else None
    pr  = average_precision_score(y_true, y_prob) if y_prob is not None else None
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  F1-Score:       {f1:.4f}")
    print(f"  Precision:      {pre:.4f}")
    print(f"  Recall:         {rec:.4f}")
    if auc: print(f"  ROC-AUC:        {auc:.4f}")
    if pr:  print(f"  PR-AUC:         {pr:.4f}")
    return {'model': name, 'f1': f1, 'precision': pre,
            'recall': rec, 'roc_auc': auc, 'pr_auc': pr}

results   = []
cv_scores = {}   # store cross-val scores for Wilcoxon test

# **Machine Learning Models**

In the Machine Learning Models I am trying  Xgboost and Random Forest Algorithms,Isolation Forest

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 1 — ISOLATION FOREST (unsupervised baseline)
# Trains on BENIGN only — no attack labels needed
# ══════════════════════════════════════════════════════════════════
print("Training Model 1: Isolation Forest...")
t0 = time.time()

iso = IsolationForest(
    n_estimators=100,
    contamination=0.1,    # expected anomaly fraction
    random_state=42,
    n_jobs=-1
)
# Train on benign training samples ONLY
iso.fit(X_train_bal[y_train_bal == 0])

# Predict: -1 = anomaly (attack), +1 = normal (benign)
iso_raw   = iso.predict(X_test)
iso_preds = (iso_raw == -1).astype(int)   # convert to 0/1
iso_score = iso.decision_function(X_test) # anomaly score
iso_prob  = 1 - (iso_score - iso_score.min()) / (iso_score.max() - iso_score.min())

print(f"Training time: {time.time()-t0:.1f}s")
r = evaluate("Isolation Forest", y_test, iso_preds, iso_prob)
results.append(r)


# ══════════════════════════════════════════════════════════════════
# MODEL 2 — RANDOM FOREST (supervised)
# ══════════════════════════════════════════════════════════════════
print("\nTraining Model 2: Random Forest...")
t0 = time.time()

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_bal, y_train_bal)

rf_preds = rf.predict(X_test)
rf_prob  = rf.predict_proba(X_test)[:, 1]

print(f"Training time: {time.time()-t0:.1f}s")
r = evaluate("Random Forest", y_test, rf_preds, rf_prob)
results.append(r)
#pickle.dump(rf, open("models/random_forest.pkl", "wb"))

# Cross-validation for Wilcoxon test (RQ1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores['Random Forest'] = cross_val_score(
    rf, X_train_bal, y_train_bal, cv=cv, scoring='f1', n_jobs=-1)
print(f"5-fold CV F1: {cv_scores['Random Forest'].mean():.4f} "
      f"± {cv_scores['Random Forest'].std():.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 3 — XGBOOST (supervised)
# ══════════════════════════════════════════════════════════════════
print("\nTraining Model 3: XGBoost...")
t0 = time.time()

scale_pos = sum(y_train_bal==0) / sum(y_train_bal==1)
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(X_train_bal, y_train_bal,
              eval_set=[(X_test, y_test)],
              verbose=False)

xgb_preds = xgb_model.predict(X_test)
xgb_prob  = xgb_model.predict_proba(X_test)[:, 1]

print(f"Training time: {time.time()-t0:.1f}s")
r = evaluate("XGBoost", y_test, xgb_preds, xgb_prob)
results.append(r)
xgb_model.save_model("models/xgboost.json")

cv_scores['XGBoost'] = cross_val_score(
    xgb_model, X_train_bal, y_train_bal, cv=cv, scoring='f1', n_jobs=-1)
print(f"5-fold CV F1: {cv_scores['XGBoost'].mean():.4f} "
      f"± {cv_scores['XGBoost'].std():.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 4 — AUTOENCODER (unsupervised deep learning)
# Trained on BENIGN only — high reconstruction error = attack
# ══════════════════════════════════════════════════════════════════
print("\nTraining Model 4: Autoencoder...")
t0 = time.time()

input_dim = X_train_bal.shape[1]

# Build autoencoder architecture
inputs   = keras.Input(shape=(input_dim,))
encoded  = keras.layers.Dense(64, activation='relu')(inputs)
encoded  = keras.layers.Dropout(0.2)(encoded)
encoded  = keras.layers.Dense(32, activation='relu')(encoded)
bottleneck = keras.layers.Dense(16, activation='relu')(encoded)
decoded  = keras.layers.Dense(32, activation='relu')(bottleneck)
decoded  = keras.layers.Dropout(0.2)(decoded)
decoded  = keras.layers.Dense(64, activation='relu')(decoded)
outputs  = keras.layers.Dense(input_dim, activation='sigmoid')(decoded)

autoencoder = keras.Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

# Train on BENIGN only
X_benign_train = X_train_bal[y_train_bal == 0]

history = autoencoder.fit(
    X_benign_train, X_benign_train,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5)
    ],
    verbose=1
)

# Compute reconstruction error
X_benign_recon = autoencoder.predict(X_benign_train, verbose=0)
benign_mse = np.mean(np.power(X_benign_train - X_benign_recon, 2), axis=1)

# Set threshold at 95th percentile of benign errors
threshold = np.percentile(benign_mse, 95)
print(f"\nAutoencoder threshold (95th percentile): {threshold:.6f}")

# Evaluate on test set
X_test_recon = autoencoder.predict(X_test, verbose=0)
test_mse     = np.mean(np.power(X_test - X_test_recon, 2), axis=1)
ae_preds     = (test_mse > threshold).astype(int)
ae_prob      = (test_mse - test_mse.min()) / (test_mse.max() - test_mse.min())

print(f"Training time: {time.time()-t0:.1f}s")
r = evaluate("Autoencoder", y_test, ae_preds, ae_prob)
results.append(r)
autoencoder.save("models/autoencoder.keras")
np.save("models/ae_threshold.npy", np.array([threshold]))

# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'],     label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.title("Autoencoder Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.savefig("results/autoencoder_training_loss.png", dpi=150)
plt.show()

# Plot reconstruction error distribution
plt.figure(figsize=(10, 5))
plt.hist(benign_mse,                          bins=100,
         alpha=0.7, color='teal',  label='Benign', density=True)
plt.hist(test_mse[y_test==1],                 bins=100,
         alpha=0.7, color='red',   label='Attack', density=True)
plt.axvline(threshold, color='black', linestyle='--',
            linewidth=2, label=f'Threshold={threshold:.4f}')
plt.title("Autoencoder Reconstruction Error Distribution")
plt.xlabel("Reconstruction Error (MSE)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig("results/autoencoder_error_dist.png", dpi=150)
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 5 — LSTM AUTOENCODER (deep learning — sequential)
# Reshapes flow records into sequences for temporal learning
# ══════════════════════════════════════════════════════════════════
print("\nTraining Model 5: LSTM Autoencoder...")
t0 = time.time()

# Reshape for LSTM: (samples, timesteps, features)
# Treat each flow record as a sequence of 1 timestep
# Group features into 6 steps of ~7 features each
n_steps    = 6
n_features = input_dim // n_steps
remainder  = input_dim % n_steps

# Pad features if not divisible
if remainder != 0:
    pad = n_steps - remainder
    X_benign_lstm = np.pad(X_benign_train, ((0,0),(0,pad)))
    X_test_lstm   = np.pad(X_test,         ((0,0),(0,pad)))
    n_features    = (input_dim + pad) // n_steps
else:
    X_benign_lstm = X_benign_train.copy()
    X_test_lstm   = X_test.copy()

X_benign_lstm = X_benign_lstm.reshape(-1, n_steps, n_features)
X_test_lstm   = X_test_lstm.reshape(-1,  n_steps, n_features)

print(f"LSTM input shape: {X_benign_lstm.shape}")

# Build LSTM Autoencoder
lstm_inputs = keras.Input(shape=(n_steps, n_features))

# Encoder
enc = keras.layers.LSTM(64, activation='tanh',
                         return_sequences=True)(lstm_inputs)
enc = keras.layers.LSTM(32, activation='tanh',
                         return_sequences=False)(enc)

# Bottleneck — repeat for decoder
rep = keras.layers.RepeatVector(n_steps)(enc)

# Decoder
dec = keras.layers.LSTM(32, activation='tanh',
                         return_sequences=True)(rep)
dec = keras.layers.LSTM(64, activation='tanh',
                         return_sequences=True)(dec)
lstm_outputs = keras.layers.TimeDistributed(
    keras.layers.Dense(n_features))(dec)

lstm_ae = keras.Model(lstm_inputs, lstm_outputs)
lstm_ae.compile(optimizer='adam', loss='mse')
lstm_ae.summary()

lstm_history = lstm_ae.fit(
    X_benign_lstm, X_benign_lstm,
    epochs=30,
    batch_size=256,
    validation_split=0.1,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5)
    ],
    verbose=1
)

# Threshold
benign_lstm_recon = lstm_ae.predict(X_benign_lstm, verbose=0)
benign_lstm_mse   = np.mean(np.power(
    X_benign_lstm - benign_lstm_recon, 2), axis=(1,2))
lstm_threshold    = np.percentile(benign_lstm_mse, 95)
print(f"\nLSTM threshold: {lstm_threshold:.6f}")

# Evaluate
test_lstm_recon = lstm_ae.predict(X_test_lstm, verbose=0)
test_lstm_mse   = np.mean(np.power(
    X_test_lstm - test_lstm_recon, 2), axis=(1,2))
lstm_preds      = (test_lstm_mse > lstm_threshold).astype(int)
lstm_prob       = ((test_lstm_mse - test_lstm_mse.min()) /
                   (test_lstm_mse.max() - test_lstm_mse.min()))

print(f"Training time: {time.time()-t0:.1f}s")
r = evaluate("LSTM Autoencoder", y_test, lstm_preds, lstm_prob)
results.append(r)
lstm_ae.save("models/lstm_autoencoder.keras")
np.save("models/lstm_threshold.npy", np.array([lstm_threshold]))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FINAL COMPARISON — ALL 5 MODELS
# ══════════════════════════════════════════════════════════════════
print("\n\n" + "="*60)
print("  MODEL COMPARISON RESULTS — ALL 5 MODELS")
print("="*60)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('f1', ascending=False)
print(results_df[['model','f1','precision','recall',
                   'roc_auc','pr_auc']].to_string(index=False))
results_df.to_csv("results/model_comparison.csv", index=False)

# ── Bar chart comparison ──────────────────────────────────────────
metrics = ['f1', 'precision', 'recall']
x       = np.arange(len(results_df))
width   = 0.25
colors  = ['#2E86AB', '#E84855', '#0F6E56']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*width, results_df[metric], width,
           label=metric.upper(), color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(results_df['model'], rotation=15, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — F1, Precision, Recall")
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("results/model_comparison_bar.png", dpi=150)
plt.show()

# ── Confusion matrices for supervised models ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, preds) in zip(axes, [
        ("Random Forest", rf_preds),
        ("XGBoost",       xgb_preds)]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, preds,
        display_labels=['BENIGN', 'ATTACK'],
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)
plt.tight_layout()
plt.savefig("results/confusion_matrices.png", dpi=150)
plt.show()

# ── Wilcoxon signed-rank test (RQ1) ──────────────────────────────
print("\n\nSTATISTICAL TESTS — RQ1 (Wilcoxon signed-rank)")
print("-"*50)
models_to_compare = list(cv_scores.keys())
for i in range(len(models_to_compare)):
    for j in range(i+1, len(models_to_compare)):
        m1, m2   = models_to_compare[i], models_to_compare[j]
        stat, p  = wilcoxon(cv_scores[m1], cv_scores[m2])
        verdict  = "SIGNIFICANT" if p < 0.05 else "not significant"
        print(f"  {m1} vs {m2}: p = {p:.4f} → {verdict}")
        print(f"    {m1} CV F1: {cv_scores[m1].mean():.4f} ± "
              f"{cv_scores[m1].std():.4f}")
        print(f"    {m2} CV F1: {cv_scores[m2].mean():.4f} ± "
              f"{cv_scores[m2].std():.4f}")

print("\n\nAll models trained and saved to /models")
print("All results saved to /results")
print("\nNext step: SHAP explainability (RQ3)")
